In [2]:
# Cell 1: Installation

# If you're in a fresh environment, install Vanna with the ChromaDB + Postgres extras:
# Remove the leading '%' if running in a plain Jupyter notebook (not IPython/Colab).
%pip install 'vanna[chromadb,postgres]'


Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: "'vanna[chromadb,postgres]'": Expected package name at the start of dependency specifier
    'vanna[chromadb,postgres]'
    ^


In [17]:
import os
from huggingface_hub import InferenceClient

# Vanna imports
from vanna.base import VannaBase
from vanna.chromadb import ChromaDB_VectorStore

class MyCustomLLM(VannaBase):
    """
    Custom LLM that calls a remote Hugging Face model via InferenceClient
    for chat completions, similar to openai_chat.py style.
    """

    def __init__(self, config=None):
        super().__init__()
        if config is None:
            raise ValueError("Config must include 'api_key' and 'model' for Hugging Face.")

        self.api_key = config.get("api_key")       # HF token
        self.model = config.get("model")          # e.g. "deepseek-ai/DeepSeek-R1"
        self.provider = config.get("provider", "together")  # If model is on 'together' or None
        self.max_tokens = config.get("max_tokens", 1000)

        if not self.api_key:
            raise ValueError("Config missing 'api_key' (HF token).")
        if not self.model:
            raise ValueError("Config missing 'model' name.")

        # Initialize Hugging Face InferenceClient
        self.client = InferenceClient(
            provider=self.provider,
            api_key=self.api_key
        )

    # -------------------------------------------------------------------------
    # Required abstract methods (from VannaBase):
    # -------------------------------------------------------------------------
    def system_message(self, message: str) -> dict:
        return {"role": "system", "content": message}

    def assistant_message(self, message: str) -> dict:
        return {"role": "assistant", "content": message}

    def user_message(self, message: str) -> dict:
        return {"role": "user", "content": message}
    # -------------------------------------------------------------------------

    def submit_prompt(self, prompt, **kwargs) -> str:
        """
        A generic method for sending [system/user/assistant] messages
        to the HF chat endpoint and returning the result text.
        """
        try:
            completion = self.client.chat.completions.create(
                model=self.model,
                messages=prompt,
                max_tokens=self.max_tokens,
                # You can add parameters like temperature=0, top_p=1, etc.
            )
            return completion.choices[0].message["content"]
        except Exception as e:
            return f"Error during inference: {e}"

    def generate_sql(self, question: str, **kwargs) -> str:
        """
        Convert a user question into *only* a final SQL query (no chain-of-thought).
        We do that by crafting a strict system message + user message.
        """
        system = self.system_message(
            "You are an expert SQL assistant. "
            "Convert the user's question into a valid SQL query. "
            "Do NOT return any chain-of-thought or reasoning. "
            "Return ONLY the SQL query."
        )
        user = self.user_message(f"Question: {question}")
        messages = [system, user]

        response_text = self.submit_prompt(messages, **kwargs)

        # Optionally parse out only the SQL if there's extra text:
        # A naive approach: if there's a code block or multiline extra text, keep best guess
        # Example: if the model returns: "SELECT * FROM table; -- no chain-of-thought"
        # We'll just strip leading/trailing whitespace
        sql_query = response_text.strip()

        # If your model sometimes includes "SQL Query:" or code fences, remove them:
        # e.g., if "```sql ...```"
        if "```" in sql_query:
            sql_query = sql_query.replace("```sql", "").replace("```", "").strip()
        if "SQL Query:" in sql_query:
            sql_query = sql_query.split("SQL Query:")[-1].strip()

        return sql_query

    def generate_explanation(self, question: str, columns, rows, **kwargs) -> str:
        """
        Use the LLM to give a short, user-friendly explanation or summary
        of the query result. This is optional (like openai_chat does).
        """
        # We build a system message that says "explain the result" without chain-of-thought
        system_prompt = (
            "You are a helpful assistant. "
            "Given the user's question and some data rows, provide a concise, direct answer. "
            "Do NOT show your chain-of-thought."
        )
        system = self.system_message(system_prompt)

        # Summarize only first N rows for safety
        n_preview = min(len(rows), 5)
        rows_preview = rows[:n_preview]

        # A naive formatting of rows
        rows_str = ""
        if rows_preview:
            rows_str = "\n".join(str(row) for row in rows_preview)
        else:
            rows_str = "No rows returned."

        user_content = (
            f"Question: {question}\n"
            f"Columns: {columns}\n"
            f"Sample rows:\n{rows_str}\n\n"
            f"Please summarize or directly answer."
        )
        user = self.user_message(user_content)

        response_text = self.submit_prompt([system, user], **kwargs)
        return response_text.strip()


In [18]:
class MyVanna(ChromaDB_VectorStore, MyCustomLLM):
    """
    Combines the ChromaDB vector store with our custom Hugging Face LLM.
    We also override `ask(...)` to replicate Vanna's typical flow:
     - retrieve context from vector store
     - generate SQL
     - optionally run SQL
     - optionally generate explanation
    """
    def __init__(self, config=None):
        ChromaDB_VectorStore.__init__(self, config=config)
        MyCustomLLM.__init__(self, config=config)

    def ask(self, question, run_sql=False, **kwargs):
        """
        1) Retrieve top N relevant docs from the vector store,
        2) Prompt the LLM to produce SQL,
        3) Optionally run that SQL on the connected DB,
        4) Optionally generate a user-friendly explanation,
        5) Return a dict with 'answer', 'sql', and possibly 'df'.
        """
        # 1) Retrieve top docs from Chroma
        # VannaBase does something similar under the hood, but let's be explicit:
        relevant_docs = self.retrieve_relevant_docs(question, limit=10)
        # You can incorporate these docs into the system prompt or do a simpler approach

        # 2) Generate SQL
        sql_query = self.generate_sql(question)

        # Prepare the default answer structure
        result = {
            "answer": None,
            "sql": sql_query,
            "df": None  # if we run SQL, we'll store DataFrame here
        }

        if not sql_query:
            result["answer"] = "No SQL could be generated."
            return result

        if run_sql:
            # 3) Execute the query
            try:
                df = self.run_sql(sql_query)
                result["df"] = df

                # 4) Optionally generate an explanation summarizing the result
                if df is not None and not df.empty:
                    columns = list(df.columns)
                    rows = df.values.tolist()
                    explanation = self.generate_explanation(question, columns, rows)
                    result["answer"] = explanation
                else:
                    # If empty, just say no rows found
                    result["answer"] = "Query returned no rows."
            except Exception as e:
                result["answer"] = f"Error executing SQL: {e}"
        else:
            # If not running SQL, just store a short note
            result["answer"] = "Generated SQL (not executed)."

        return result


In [19]:
# Cell 3: Create an instance of MyVanna and connect to Postgres
API_KEY = os.getenv("HUGGINGFACE_API_KEY")  # Your Hugging Face token
# Provide your own config for the Hugging Face model
custom_llm_config = {
    "api_key": API_KEY,  # Your Hugging Face token
    "model": "deepseek-ai/DeepSeek-R1",        # Example model name
    "provider": "together",                    # or None if not using "together"
    "max_tokens": 1024
}

vn = MyVanna(config=custom_llm_config)

# Connect to Postgres (replace with actual credentials!)
import os
vn.connect_to_postgres(host=os.getenv('DB_HOST'), dbname=os.getenv('DB_NAME'), user='postgres', password=os.getenv('DB_PASSWORD'), port=5432)


print("Connected to Postgres successfully.")


Connected to Postgres successfully.


In [20]:
# Cell 4: Extract schema information

# Include table_catalog so that Vanna can find the 'database' column
df_tables = vn.run_sql("""
SELECT
    table_catalog,
    table_schema,
    table_name
FROM INFORMATION_SCHEMA.TABLES
WHERE table_type = 'BASE TABLE'
  AND table_schema NOT IN ('pg_catalog', 'information_schema');
""")

# display(df_tables)

df_fk = vn.run_sql("""
SELECT
  tc.table_catalog,               -- add table_catalog here, too!
  tc.table_schema,
  tc.table_name,
  kcu.column_name,
  ccu.table_schema AS foreign_table_schema,
  ccu.table_name AS foreign_table_name,
  ccu.column_name AS foreign_column_name
FROM information_schema.table_constraints AS tc
JOIN information_schema.key_column_usage AS kcu
  ON tc.constraint_name = kcu.constraint_name
  AND tc.table_schema = kcu.table_schema
JOIN information_schema.constraint_column_usage AS ccu
  ON ccu.constraint_name = tc.constraint_name
  AND ccu.table_schema = tc.table_schema
WHERE tc.constraint_type = 'FOREIGN KEY'
  AND tc.table_schema NOT IN ('pg_catalog', 'information_schema');
""")

# display(df_fk)




In [ ]:
# Suppose df_tables has columns: [table_catalog, table_schema, table_name]
df_tables_for_docs = df_tables.copy()

# Create a single text column that describes each table (doc lines)
df_tables_for_docs["documentation"] = df_tables_for_docs.apply(
    lambda row: (
        f"Database: {row['table_catalog']} / Schema: {row['table_schema']} / "
        f"Table: {row['table_name']}"
    ),
    axis=1
)

# We only keep the 'documentation' column so Vanna doesn't see anything else
df_tables_for_docs = df_tables_for_docs[["documentation"]]

# Generate a training plan from these doc lines
for index, row in df_tables.iterrows():
    doc_line = f"Database: {row['table_catalog']} / Schema: {row['table_schema']} / Table: {row['table_name']}"
    vn.train(documentation=doc_line)


In [ ]:
df_fk_for_docs = df_fk.copy()

df_fk_for_docs["documentation"] = df_fk_for_docs.apply(
    lambda row: (f"Foreign Key: {row['table_schema']}.{row['table_name']} "
                 f"({row['column_name']}) => {row['foreign_table_schema']}.{row['foreign_table_name']}({row['foreign_column_name']})"),
    axis=1
)
df_fk_for_docs = df_fk_for_docs[["documentation"]]
for index, row in df_fk.iterrows():
    doc_line = f"Database: {row['table_catalog']} / Schema: {row['table_schema']} / Table: {row['table_name']}"
    vn.train(documentation=doc_line)


In [ ]:
vn.train(documentation="All the Captains, Agents, and Organization related people are in User table with different userTypes, and UserType is another Table which specify type of user. and Customers table consists of registered users, passengers table consists of passenegers who travelled with us and need not to be in customers.")
vn.train(documentation="The User table has all the information about the users, and the UserType table has the type of users. The Customers table has the information about the registered users, and the Passengers table has the information about the passengers who have traveled with us.")
vn.train(documentation="ReservedTickets table consists of all booked tickets and trip details, passenger details")
vn.train(documentation="TicketTransaction table consists of all the transactions related to tickets, and this table has all the payment details with necessary columns of what platform and when it got booked")
vn.train(documentation="ReservedTicketSeats table consists of and triplevel details, seatIds, passenger details of seat and so on")
vn.train(documentation="AgentPaymentTransaction table consists of all the transactions related to agents, and this table has all the payment details with necessary columns of what platform and when it got booked")

In [15]:

with open("database_documentation.txt", "r") as f:
    db_docs = f.read()

vn.add_documentation(db_docs, doc_name="DB Schema Docs")
# Then your user queries can reference it


'e2ec4cc6-0baf-599a-b7e6-76f557811b9d-doc'

In [24]:
from vanna.flask import VannaFlaskApp
app = VannaFlaskApp(vn)
app.run()

Your app is running at:
http://localhost:8084
 * Serving Flask app 'vanna.flask'
 * Debug mode: on
